# detect

> Read what you paste into the Start box, and say what abhyasa will do with it. No network call.

The Learn page has one box to start something new. You paste a link, a paper, a topic, a file name or some text. As you type,
one quiet line says what abhyasa will do. When two readings are both likely, the box asks one short question. Options show
only when they change the result.

`detect` makes this reading from the text only. It makes no network call and it resolves no host name. The one lookup is a
match against the titles of your vault documents, and the caller does that lookup (`vault_title`).

In [ ]:
#| default_exp detect

In [ ]:
#| hide
import os, tempfile
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

Import the link rules of `sources`, so the box and a walk from a link agree on what a paper and a local address are.

In [ ]:
#| export
import re, unicodedata
from pathlib import Path
from urllib.parse import urlparse
from fastcore.utils import *
from abhyasa.sources import ARXIV_ID, ARXIV_LINK, as_link, is_paper, _local
from abhyasa.settings import val

## The words

Each reading has one quiet line. The numbers in the lines come from the settings, so a line is never false.

In [ ]:
#| export
HINT = 'Not sure what to paste? Try a topic, such as how attention works.'
LINES = dict(
    arxiv='An arXiv paper. abhyasa reads it in full and makes a walk of up to {steps} ideas.',
    pdf_link='A PDF link. abhyasa reads it in full and makes a walk of up to {steps} ideas.',
    pdf='A PDF. abhyasa reads it in full.',
    repo='A GitHub repository. abhyasa makes a tool walk. Code steps open in Leela.',
    site='A docs site. abhyasa reads the pages in this folder. It reads at most {pages} pages.',
    topic='A topic. The tutor plans a walk and looks for sources on the web.',
    own='A topic. The tutor plans a walk from its own knowledge. Passages are not word for word.',
    text='Pasted text, {lines}. abhyasa finds the ideas in it and makes a walk.',
    file='A text file. abhyasa finds the ideas in it.',
    doc='A document in your vault. abhyasa reads it word for word and makes a walk.',
    video='Video links do not work yet.',
    local='abhyasa does not read links to this computer or a local network.',
    folder='A folder on this computer. abhyasa cannot read folders yet.',
    sources='{n} sources. abhyasa cannot start from more than one source yet.')

## Folding a name

A title matches when it is the same text with case and marks folded, so `Maghā` finds `magha`. Only case and marks fold:
`nakṣatras` does not find `Nakshatras`. This is the fold of the concepts list.

In [ ]:
#| export
def fold(s) -> str:
    "`s` in lower case with no marks, and with the spaces at its ends removed."
    return ''.join(ch for ch in unicodedata.normalize('NFKD', str(s or '')) if not unicodedata.combining(ch)).lower().strip()

In [ ]:
test_eq(fold('  Maghā '), 'magha')

## One source

`source_of` reads one word as a source: a paper, a GitHub repository, a docs site, a video, a local link or a folder. A word
with no shape of a source gives None. A bare `github.com/...`, or a name with a dot and a slash, gets `https://`. The video
hosts are the video hosts of `research.SOCIAL`. A folder is a path that starts with `/` or `~`, or a folder name in `folders`.

In [ ]:
#| export
VIDEO = ('youtube.com', 'youtu.be', 'vimeo.com')
GITHUB = ('github.com', 'www.github.com')
BARE = re.compile(r'[\w-]+(?:\.[\w-]+)+/\S*')   # a link with no scheme, such as github.com/a/b or docs.python.org/3/

def as_url(s:str) -> str|None:
    "`s` as a web link: as it is, with `https://` for a bare link, or None."
    if re.fullmatch(r'https?://\S+', s, re.I): return s
    return 'https://' + s if BARE.fullmatch(s) else None

def source_of(s:str, folders=()) -> tuple|None:
    "The kind of the one word `s` as a source, and its link or path; None when `s` does not have the shape of a source."
    s = s.strip()
    if re.match(r'arxiv:', s, re.I) or ARXIV_ID.fullmatch(s) or ARXIV_LINK.fullmatch(s): return 'paper', as_link(s) or s
    if s.startswith(('/', '~')) or (s and fold(s) in {fold(f) for f in folders}): return 'folder', s
    if s.lower().startswith('file:'): return 'local', s
    if not (u := as_url(s)): return None
    try: host = (urlparse(u).hostname or '').lower()
    except ValueError: return None
    if not host: return None
    if any(host == h or host.endswith('.' + h) for h in VIDEO): return 'video', u
    if _local(host): return 'local', u
    if is_paper(u): return 'paper', u
    if host in GITHUB and len([p for p in urlparse(u).path.split('/') if p]) >= 2: return 'repo', u
    return 'site', u

A word with no shape of a source, such as a topic, gives None. `_local` resolves no name, so `printer.local` is local with no lookup.

In [ ]:
test_eq(source_of('github.com/a/b'), ('repo', 'https://github.com/a/b'))
test_eq(source_of('arxiv:1706.03762'), ('paper', '1706.03762'))
test_eq(source_of('~/code/course22')[0], 'folder')
test_eq(source_of('course22', folders=('course22',))[0], 'folder')
test_eq(source_of('http://printer.local/')[0], 'local')
test_eq(source_of('attention'), None)

## The reading

`detect` gives an `AttrDict`:

- `kind`: one of `empty paper repo site doc topic text ask video local folder sources`.
- `line`: the quiet line.
- `ask`: None, or the question, with two answers as (label, kind).
- `options`: the fields that matter, from `pages tool title subject find`.
- `send`: the fields of `/walk/new` for this reading.
- `sources`: for `sources`, each source as (kind, link or path). Task 6a files them.

`pick` is the answer to a question: `doc`, `topic` or `text`. `file` is the name of a dropped file: a `.pdf` is in the vault
under the name in the box, and a `.md` or `.txt` is in the box as text.

In [ ]:
#| export
def _reading(kind, line, ask=None, options=(), send=None, sources=()) -> AttrDict:
    return AttrDict(kind=kind, line=line, ask=ask, options=tuple(options), send=dict(send or {}), sources=tuple(sources))

def _n(n, w): return f"{n} {w}{'' if n == 1 else 's'}"

def _topic(t, research): return _reading('topic', LINES['topic' if research else 'own'], send=dict(topic=t))

def _text(t, lines): return _reading('text', LINES['text'].format(lines=_n(lines, 'line')), options=('title', 'subject', 'find'), send=dict(text=t))

def _source(kind, u, pages):
    "The reading of one source of `kind` at `u`."
    if kind == 'paper':
        return _reading('paper', LINES['arxiv' if 'arxiv.org/' in u or not u.startswith('http') else 'pdf_link'].format(steps=val('walks.steps')), send=dict(url=u))
    if kind == 'repo': return _reading('repo', LINES['repo'], options=('tool',), send=dict(url=u, tool='on'))
    if kind == 'site': return _reading('site', LINES['site'].format(pages=pages), options=('pages',), send=dict(url=u))
    return _reading(kind, LINES[kind])

In [ ]:
#| export
def detect(text:str, titles=(), research:bool=True, pages:int=8, file:str='', folders=(), pick:str='') -> AttrDict:
    "What abhyasa does with `text` from the Start box. `titles`: vault titles to match; `folders`: folder names that you named."
    t, f = (text or '').strip(), (file or '').lower()
    if t and f.endswith('.pdf'): return _reading('paper', LINES['pdf'], send=dict(ref=t))
    if t and f.endswith(('.md', '.txt')): return _reading('text', LINES['file'], options=('title', 'subject'), send=dict(text=t, title=Path(file).stem))
    if not t: return _reading('empty', HINT)
    lines, words = [l for l in t.splitlines() if l.strip()], t.split()
    parts = [p for p in re.split(r'[\s,]+', t) if p]
    if len(parts) > 1 and all(srcs := [source_of(p, folders) for p in parts]):
        return _reading('sources', LINES['sources'].format(n=len(srcs)), sources=srcs)
    if len(lines) == 1 and (s := source_of(t if len(words) == 1 or t.startswith(('/', '~')) else '', folders)): return _source(*s, pages)
    if len(lines) > 1 or len(t) > 200: return _text(t, len(lines))
    hit = next((x for x in titles if fold(x) == fold(t)), None) if len(words) <= 6 else None
    if hit:
        if pick == 'doc': return _reading('doc', LINES['doc'], send=dict(ref=hit))
        if pick == 'topic': return _topic(t, research)
        q = f'{hit} is in your vault.'
        return _reading('ask', q, ask=AttrDict(q=q, a=('Walk through that document', 'doc'), b=('Plan a walk on the topic', 'topic')))
    if 7 <= len(words) <= 12 and t.endswith('.'):
        if pick == 'topic': return _topic(t, research)
        if pick == 'text': return _text(t, 1)
        q = 'Is this a topic or a text?'
        return _reading('ask', q, ask=AttrDict(q=q, a=('A topic', 'topic'), b=('A text', 'text')))
    return _text(t, 1) if t.endswith('.') and len(words) > 12 else _topic(t, research)

## The cases

`DETECT_CASES` holds each shape that the box knows, with the kind it must give. The tests of the app use the same table.

In [ ]:
#| export
DETECT_CASES = [('1706.03762', (), 'paper'), ('arxiv:1706.03762v5', (), 'paper'), ('  https://arxiv.org/abs/1706.03762  ', (), 'paper'),
                ('https://x.org/a.pdf', (), 'paper'), ('github.com/vedicreader/fossick', (), 'repo'), ('https://github.com/a/b', (), 'repo'),
                ('https://docs.python.org/3/library/asyncio.html', (), 'site'), ('Nakshatras', ('Nakshatras',), 'ask'),
                ('how attention works', (), 'topic'), ('nakṣatra', (), 'topic'), ('a\nb', (), 'text'), ('x' * 300, (), 'text'),
                ('The moon spends about one day in each of the lunar mansions.', (), 'ask'), ('https://youtu.be/x', (), 'video'),
                ('http://127.1/', (), 'local'), ('http://printer.local/', (), 'local'), ('', (), 'empty'), ('   ', (), 'empty'),
                ('~/code/course22', (), 'folder'), ('/Users/me/My notes', (), 'folder'),
                ('github.com/a/b ~/code/fastbook', (), 'sources'), ('1706.03762\n2005.14165', (), 'sources')]

Each case gives its kind.

In [ ]:
for t, ti, k in DETECT_CASES: test_eq(detect(t, ti).kind, k)

A vault title asks first. Its first answer walks through the document, and the answer `doc` sends its title as `ref`.

In [ ]:
r = detect('Nakshatras', ('Nakshatras',))
test_eq(r.ask.a, ('Walk through that document', 'doc'))
test_eq(r.ask.q, 'Nakshatras is in your vault.')
test_eq(detect('nakshatras', ('Nakshatras',), pick='doc').send, dict(ref='Nakshatras'))
test_eq(detect('Nakshatras', ('Nakshatras',), pick='topic').kind, 'topic')

With research off, the topic line says that the tutor uses its own knowledge.

In [ ]:
assert 'own knowledge' in detect('how attention works', research=False).line
test_eq(detect('how attention works').send, dict(topic='how attention works'))

Pasted text shows Title, Subject and Find concepts only. A sentence of 7 to 12 words answered as a text is one line of text.

In [ ]:
test_eq(detect('a\nb').options, ('title', 'subject', 'find'))
test_eq(detect('a\nb').line, 'Pasted text, 2 lines. abhyasa finds the ideas in it and makes a walk.')
test_eq(detect('The moon spends about one day in each of the lunar mansions.', pick='text').kind, 'text')

A repo ticks the tool box. A site says how many pages it reads. A paper says how many ideas a walk has at most.

In [ ]:
r = detect('github.com/vedicreader/fossick')
test_eq((r.options, r.send), (('tool',), dict(url='https://github.com/vedicreader/fossick', tool='on')))
test_eq(detect('https://docs.python.org/3/', pages=5).line, 'A docs site. abhyasa reads the pages in this folder. It reads at most 5 pages.')
test_eq(detect('1706.03762').line, 'An arXiv paper. abhyasa reads it in full and makes a walk of up to 12 ideas.')
test_eq(detect('arxiv.org/abs/1706.03762').send, dict(url='https://arxiv.org/abs/1706.03762'))

A dropped file is certain: the box never asks for it.

In [ ]:
test_eq(detect('Nakshatras', ('Nakshatras',), file='naks.pdf').line, 'A PDF. abhyasa reads it in full.')
r = detect('Line one.\nLine two.', file='notes.md')
test_eq((r.kind, r.line, r.options, r.send['title']), ('text', 'A text file. abhyasa finds the ideas in it.', ('title', 'subject'), 'notes'))

Several sources at once give each source. A folder name that you named is a folder.

In [ ]:
test_eq(detect('github.com/a/b, ~/code/fastbook').sources, (('repo', 'https://github.com/a/b'), ('folder', '~/code/fastbook')))
test_eq(detect('course22', folders=('course22',)).kind, 'folder')

## The vault title

`vault_title` finds the one document on the main shelf whose title is the text, with case and marks folded. It skips your walks,
talks and concepts. A text of more than 6 words never matches.

In [ ]:
#| export
SKIP_KINDS = ('journey', 'talk', 'concept')

def vault_title(vault, q:str) -> str|None:
    "The title of the document on `vault` (the main shelf) that is `q`, with case and marks folded; None when there is none."
    f = fold(q)
    if not f or len(f.split()) > 6: return None
    rows = vault.t.docs(select='title', where=f"kind is null or kind not in ({','.join('?' * len(SKIP_KINDS))})", where_args=list(SKIP_KINDS))
    return next((r['title'] for r in rows if fold(r['title']) == f), None)

A test vault with one document named Nakshatras. A part of the title does not match.

In [ ]:
from abhyasa.store import temp_learning

In [ ]:
lr = temp_learning()
lr.vault.add('Some text about nakshatras.', title='Nakshatras', source='test://n')
test_eq(vault_title(lr.vault, ' NAKSHATRAS '), 'Nakshatras')
test_eq(vault_title(lr.vault, 'naksha'), None)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()